In [45]:
import os
import certifi

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool
import requests

In [46]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [47]:
# LOAD ENVIRONMENT VARIABLES
load_dotenv()

True

In [48]:
search_tool = DuckDuckGoSearchRun()

In [49]:
@tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a given city using the OpenWeatherMap API.
    """
    url = f"http://api.weatherstack.com/current?access_key={os.getenv('WEATHERSTACK_API_KEY')}&query={city}"
    response = requests.get(url)

    data = response.json()
    if "current" not in data:
        return f"Error fetching weather data for {city}: {data.get('error', {})}"
    
    return f"The current weather in {city} is {data['current']['temperature']}°C with {data['current']['weather_descriptions'][0]}. and Humidity is {data['current']['humidity']}%"

In [33]:
result = search_tool.invoke("What is the capital of France?")
result

'1 day ago - Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km2 (40.7 sq mi), and a metropolitan population of 13.2 million as of January 2026. Located on the river Seine in the centre of the Île-de-France region, it is the largest ... 5 days ago - Its metropolitan area extends from ... and the North Sea. Its 18 integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi), with a total population estimated at over 69.1 million in 2026. Its capital, largest city and main cultural and economic centre is Paris, with a metropolitan population of over 13 million. Metropolitan France was settled ... July 2, 2026 - This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944. 2 days ago - The capital of France is Paris. Enjoy the videos and music you love, upload original content, and share it all with friends, family, and the worl

In [50]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [15]:
response = llm.invoke("What year is it ?")
response

AIMessage(content="I'm not currently aware of the year. I'm a large language model, I don't have have access to real-time information, and my knowledge cutoff is December 2023.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 40, 'total_tokens': 78, 'completion_time': 0.101680976, 'completion_tokens_details': None, 'prompt_time': 0.011180921, 'prompt_tokens_details': None, 'queue_time': 0.133682463, 'total_time': 0.112861897}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc78c-a50b-7bf1-9263-0ec99bae1c16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 38, 'total_tokens': 78})

In [51]:
from langsmith import Client

In [52]:
client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)

In [53]:
tools = [search_tool, get_weather]

In [54]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

In [55]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [56]:
response = agent_executor.invoke(
    {
        "input": (
            "Find the capital of DRCongo",
            "and then find its current weather."
        )
    }
)



> Entering new AgentExecutor chain...
To answer this question, I first need to find the capital of the Democratic Republic of Congo (DRCongo) and then find its current weather.

Action: duckduckgo_search
Action Input: capital of DRCongoJan 15, 2026 ... Kinshasa formerly named Léopoldville (Dutch: Leopoldstad ) from 1881 to 1966, is the capital and largest city of the Democratic Republic of the Congo. Dec 14, 2025 ... Kinshasa is the capital city of DRC, and also the world's second -largest French speaking city in the world Music is a major export for the DRC # ... May 5, 2026 ... Thousands of pro-government supporters took to the streets of the Congolese capital, Kinshasa, on Monday in support of United States sanctions against ... Mar 28, 2026 ... You will not watch this on your tv.Kinshasa DRC #DRCongo #DRC #. 00:22 ... The Capital City of Congo DRC . They're still in 1973 .Good Music. 01:58. The ... 4 days ago ... The country is often referred to by its acronym, the DRC, or called

In [57]:
print(response["output"])

The capital of DRCongo is Kinshasa, and its current weather is 36°C with Smoky haze and a humidity of 30%.
